## Coordinated Behavior Detection

In [ ]:
## Real fraud is rarely solo. A single person buying fake likes is a nuisance. 
# A bot farm with 500 accounts hitting 85 videos simultaneously is organized crime. 
# We build three signals to detect coordination, then combine them into a single score.

## Signal 1: Peak Hour Clustering
#Bot farms often run on schedules -- same timezone, same shift, or same automated trigger time. We find hours where an
#unusually high number of suspicious videos peaked.

## Signal 2: Temporal Co-Spiking
# If 10 unrelated videos all had their maximum like spike within the same hour window, that's not coincidence. 
# That's a coordinated attack.

## Signal 3: Bot Network Clustering (DBSCAN)
# We take ONLY the 500 suspicious videos and cluster them by engagement style. Videos in the same cluster share the same fake
#engagement fingerprint -- suggesting they're from the same bot network using identical settings.

In [11]:
import pandas as pd
from sklearn.cluster import DBSCAN
from collections import Counter
import seaborn as sns
from sklearn.preprocessing import StandardScaler


In [8]:
yt_sample = pd.read_csv("data/processed/yt_sample.csv")

In [15]:
video_features = pd.read_csv("data/processed/engineered_video_features2.csv")

In [ ]:
# ============================================================
# 1. PEAK HOUR CLUSTERING — Bot farms may often spike on schedule
# ============================================================

In [16]:
def detect_peak_hour_clusters(df, min_cluster_size=5):
    """
    Groups videos by peak_hour. Large clusters at odd hours (3AM, 4AM) 
    suggest coordinated bot activity.
    """
    peak_counts = df['peak_hour'].value_counts()
    
    # Hours with unusually high activity
    mean_peaks = peak_counts.mean()
    std_peaks = peak_counts.std()
    suspicious_hours = peak_counts[peak_counts > mean_peaks + 1.5 * std_peaks].index.tolist()
    
    df['is_suspicious_hour'] = df['peak_hour'].isin(suspicious_hours)
    
    # Size of each peak hour cluster
    hour_cluster_size = df.groupby('peak_hour').size().to_dict()
    df['peak_hour_cluster_size'] = df['peak_hour'].map(hour_cluster_size)
    
    return df, suspicious_hours

video_features, suspicious_hours = detect_peak_hour_clusters(video_features)
print(f"Suspicious peak hours (possible bot schedule): {sorted(suspicious_hours)}")
print(video_features['peak_hour'].value_counts().head(10))

Suspicious peak hours (possible bot schedule): [12]
peak_hour
12    889
21    752
20    688
22    675
11    616
19    582
0     524
16    520
23    502
17    495
Name: count, dtype: int64


In [ ]:
# ============================================================
# 2. ENGAGEMENT PATTERN SIMILARITY — Same bot network signature
# ============================================================

# Bot farms often use identical settings across all their videos.
# This creates tight clusters in engagement ratio space.

In [12]:
def detect_engagement_clusters_v2(df, eps=0.3, min_samples=5):
    """
    Use MORE features and standardize them first.
    Cluster on: like_view_ratio, comment_view_ratio, sustainability, like_spike_ratio
    """
    cluster_features = ['like_view_ratio', 'comment_view_ratio', 'sustainability', 'like_spike_ratio']
    X = df[cluster_features].fillna(0)
    
    # CRITICAL: Standardize so all features contribute equally
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # eps=0.3 in standardized space = ~0.3 standard deviations apart
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(X_scaled)
    df['engagement_cluster'] = clustering.labels_
    df['in_engagement_cluster'] = df['engagement_cluster'] != -1
    
    # Only count real clusters (not noise)
    cluster_sizes = df[df['engagement_cluster'] != -1].groupby('engagement_cluster').size()
    df['engagement_cluster_size'] = df['engagement_cluster'].map(cluster_sizes).fillna(0)
    
    return df, cluster_sizes

video_features, engagement_clusters = detect_engagement_clusters_v2(video_features)
print(f"Found {len(engagement_clusters)} clusters")
print(f"Cluster sizes:\n{engagement_clusters.sort_values(ascending=False).head(10)}")

Found 9 clusters
Cluster sizes:
engagement_cluster
0    9534
1      13
2       6
7       6
5       6
6       5
3       5
4       4
8       3
dtype: int64


In [ ]:
# ============================================================
#  Only cluster the SUSPICIOUS subset (real T&S workflow)
# ============================================================
# The above clustered ALL videos, creating one giant blob of 9,534 normal videos.
# Real T&S teams only cluster their INVESTIGATION QUEUE.

In [18]:
def detect_engagement_clusters_v2(df, eps=0.3, min_samples=5):
    """
    Use MORE features and standardize them first.
    Cluster on: like_view_ratio, comment_view_ratio, sustainability, like_spike_ratio
    """
    cluster_features = ['like_view_ratio', 'comment_view_ratio', 'sustainability', 'like_spike_ratio']
    X = df[cluster_features].fillna(0)
    
    # CRITICAL: Standardize so all features contribute equally
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # eps=0.3 in standardized space = ~0.3 standard deviations apart
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(X_scaled)
    df['engagement_cluster'] = clustering.labels_
    df['in_engagement_cluster'] = df['engagement_cluster'] != -1
    
    # Only count real clusters (not noise)
    cluster_sizes = df[df['engagement_cluster'] != -1].groupby('engagement_cluster').size()
    df['engagement_cluster_size'] = df['engagement_cluster'].map(cluster_sizes).fillna(0)
    
    return df, cluster_sizes

video_features, engagement_clusters = detect_engagement_clusters_v2(video_features)
print(f"Found {len(engagement_clusters)} clusters")
print(f"Cluster sizes:\n{engagement_clusters.sort_values(ascending=False).head(10)}")

Found 9 clusters
Cluster sizes:
engagement_cluster
0    9534
1      13
2       6
7       6
5       6
6       5
3       5
4       4
8       3
dtype: int64


In [19]:
def detect_engagement_clusters_v2(df, eps=0.3, min_samples=5):
    """
    Use MORE features and standardize them first.
    Cluster on: like_view_ratio, comment_view_ratio, sustainability, like_spike_ratio
    """
    cluster_features = ['like_view_ratio', 'comment_view_ratio', 'sustainability', 'like_spike_ratio']
    X = df[cluster_features].fillna(0)
    
    # CRITICAL: Standardize so all features contribute equally
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # eps=0.3 in standardized space = ~0.3 standard deviations apart
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(X_scaled)
    df['engagement_cluster'] = clustering.labels_
    df['in_engagement_cluster'] = df['engagement_cluster'] != -1
    
    # Only count real clusters (not noise)
    cluster_sizes = df[df['engagement_cluster'] != -1].groupby('engagement_cluster').size()
    df['engagement_cluster_size'] = df['engagement_cluster'].map(cluster_sizes).fillna(0)
    
    return df, cluster_sizes

video_features, engagement_clusters = detect_engagement_clusters_v2(video_features)
print(f"Found {len(engagement_clusters)} clusters")
print(f"Cluster sizes:\n{engagement_clusters.sort_values(ascending=False).head(10)}")

Found 9 clusters
Cluster sizes:
engagement_cluster
0    9534
1      13
2       6
7       6
5       6
6       5
3       5
4       4
8       3
dtype: int64


In [20]:
# ============================================================
# FIXED: Only cluster the SUSPICIOUS subset (real T&S workflow)
# ============================================================


In [21]:
def detect_bot_networks(df, eps=0.5, min_samples=3):
    """
    Cluster ONLY videos already flagged as suspicious.
    Normal videos are left as noise (-1).
    This finds 'families' of abuse within the investigation queue.
    """
    suspicious_mask = df['is_suspicious_v2'] == True
    suspicious_df = df[suspicious_mask].copy()
    
    print(f"Investigation queue size: {len(suspicious_df)} videos")
    
    if len(suspicious_df) < 10:
        print("Too few suspicious videos for meaningful clustering")
        df['bot_network_id'] = -1
        df['in_bot_network'] = False
        df['bot_network_size'] = 0
        return df, pd.Series()
    
    # Features that bot networks share: engagement style + timing + sustainability
    cluster_features = ['like_view_ratio', 'comment_view_ratio', 'sustainability', 'like_spike_ratio', 'peak_hour']
    X = suspicious_df[cluster_features].fillna(0)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # eps=0.5 in standardized space = moderate separation
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(X_scaled)
    suspicious_df['bot_network_id'] = clustering.labels_
    
    # Merge back to full dataframe
    df['bot_network_id'] = -1
    df.loc[suspicious_df.index, 'bot_network_id'] = suspicious_df['bot_network_id']
    
    df['in_bot_network'] = (df['bot_network_id'] != -1) & (df['is_suspicious_v2'] == True)
    
    # Network sizes (only for suspicious videos in real clusters)
    network_sizes = suspicious_df[suspicious_df['bot_network_id'] != -1].groupby('bot_network_id').size()
    df['bot_network_size'] = df['bot_network_id'].map(network_sizes).fillna(0)
    
    print(f"\nFound {len(network_sizes)} bot networks")
    print(f"Network sizes:\n{network_sizes.sort_values(ascending=False)}")
    print(f"\nVideos in networks: {df['in_bot_network'].sum()}")
    print(f"Solo suspicious videos (no network): {(suspicious_mask & (df['bot_network_id'] == -1)).sum()}")
    
    return df, network_sizes

video_features, bot_networks = detect_bot_networks(video_features)

Investigation queue size: 500 videos

Found 15 bot networks
Network sizes:
bot_network_id
0     94
5     64
3     64
7     11
10     7
4      6
13     6
1      5
9      5
6      4
2      4
14     4
8      3
12     3
11     3
dtype: int64

Videos in networks: 283
Solo suspicious videos (no network): 217


In [7]:
# ============================================================
# CELL 7: TEMPORAL CO-SPIKING
# ============================================================

In [24]:
yt_sample["created_at"] = pd.to_datetime(yt_sample["created_at"])


In [25]:
# We need to go back to yt_sample (time-series data) for this
def detect_cospiking(df_raw, window_hours=2):
    """
    Finds videos that had their max like spike within the same hour window.
    Bot networks often hit multiple videos simultaneously.
    """
    results = []
    
    for video_id, group in df_raw.groupby("video_id"):
        if len(group) < 2:
            continue
            
        group = group.sort_values("created_at")
        like_growth = group["likes"].diff().fillna(0)
        
        # Find the exact timestamp of max spike
        if like_growth.max() > 0:
            spike_idx = like_growth.idxmax()
            spike_time = group.loc[group.index == spike_idx, "created_at"].iloc[0]
            spike_value = like_growth.max()
            
            results.append({
                "video_id": video_id,
                "spike_time": spike_time,
                "spike_hour": spike_time.floor('h'),
                "spike_value": spike_value
            })
    
    spike_df = pd.DataFrame(results)
    
    # Count how many videos spiked in each hour window
    spike_counts = spike_df.groupby('spike_hour').size()
    spike_df['cospike_count'] = spike_df['spike_hour'].map(spike_counts)
    
    # Videos that spiked alongside 5+ others in same hour = coordinated
    spike_df['is_cospike'] = spike_df['cospike_count'] >= 5
    
    return spike_df

print("\nDetecting temporal co-spikes...")
spike_df = detect_cospiking(yt_sample)
print(f"Videos in co-spike windows: {spike_df['is_cospike'].sum()}")

# Merge back to video_features
video_features = video_features.merge(
    spike_df[['video_id', 'cospike_count', 'is_cospike']], 
    on='video_id', 
    how='left'
)
video_features['is_cospike'] = video_features['is_cospike'].fillna(False)
video_features['cospike_count'] = video_features['cospike_count'].fillna(0)


Detecting temporal co-spikes...
Videos in co-spike windows: 307


In [26]:
video_features.to_csv("engineered_video_features3.csv", index=False)